In [1]:
import pandas as pd
import numpy as np
import joblib
import re
import string

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

print("Libraries imported successfully!")

Libraries imported successfully!


In [2]:
svm_model = joblib.load(
    "../model/svm_model.pkl"
)

tfidf = joblib.load(
    "../model/tfidf_vectorizer.pkl"
)

print("Final model and TF-IDF vectorizer loaded!")

Final model and TF-IDF vectorizer loaded!


In [3]:
stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

In [4]:
def clean_text(text):
    
    text = str(text)
    
    # Lowercase
    text = text.lower()
    
    # Remove URLs
    text = re.sub(
        r"http\S+|www\S+|https\S+",
        "",
        text
    )
    
    # Remove HTML
    text = re.sub(
        r"<.*?>",
        "",
        text
    )
    
    # Remove punctuation
    text = text.translate(
        str.maketrans(
            "",
            "",
            string.punctuation
        )
    )
    
    # Remove numbers
    text = re.sub(
        r"\d+",
        "",
        text
    )
    
    # Remove extra spaces
    text = re.sub(
        r"\s+",
        " ",
        text
    ).strip()
    
    # Tokenization
    words = text.split()
    
    # Stopword removal + lemmatization
    words = [
        lemmatizer.lemmatize(word)
        for word in words
        if word not in stop_words
    ]
    
    return " ".join(words)

In [5]:
def predict_news(news_text):
    
    # Clean input
    cleaned_text = clean_text(news_text)
    
    # Convert to TF-IDF
    text_vector = tfidf.transform(
        [cleaned_text]
    )
    
    # Predict
    prediction = svm_model.predict(
        text_vector
    )[0]
    
    # Convert label
    if prediction == 0:
        result = "FAKE NEWS"
    else:
        result = "REAL NEWS"
    
    return result

In [6]:
news = """
The government announced a new education policy
that will provide additional funding for schools
and improve digital learning facilities.
"""

result = predict_news(news)

print("Prediction:", result)

Prediction: FAKE NEWS


In [7]:
news = """
Scientists have discovered a secret technology
that allows ordinary people to live for 500 years.
The technology will be released to everyone next week.
"""

result = predict_news(news)

print("Prediction:", result)

Prediction: FAKE NEWS


In [10]:
def predict_news_with_score(news_text):
    
    cleaned_text = clean_text(news_text)
    
    text_vector = tfidf.transform(
        [cleaned_text]
    )
    
    prediction = svm_model.predict(
        text_vector
    )[0]
    
    decision_score = svm_model.decision_function(
        text_vector
    )[0]
    
    if prediction == 0:
        result = "FAKE NEWS"
    else:
        result = "REAL NEWS"
    
    return result, decision_score

In [9]:
news = """
The government announced a new program
to improve digital education in rural schools.
"""

result, score = predict_news_with_score(news)

print("Prediction:", result)
print("Decision score:", round(score, 4))

Prediction: FAKE NEWS
Decision score: -0.2089


In [11]:
test_news = [
    """
    The national education department announced
    a new scholarship program for university students.
    """,
    
    """
    Scientists have discovered a magical device
    that can make humans invisible for several hours.
    """,
    
    """
    The central bank announced changes to its
    monetary policy following its scheduled meeting.
    """,
    
    """
    A viral message claims that drinking a special
    liquid can make people immune to every disease.
    """
]

In [12]:
for i, news in enumerate(test_news, start=1):
    
    result, score = predict_news_with_score(news)
    
    print(f"News {i}")
    print("Prediction:", result)
    print("Score:", round(score, 4))
    print("-" * 50)

News 1
Prediction: FAKE NEWS
Score: -0.6756
--------------------------------------------------
News 2
Prediction: FAKE NEWS
Score: -0.6802
--------------------------------------------------
News 3
Prediction: FAKE NEWS
Score: -0.2673
--------------------------------------------------
News 4
Prediction: FAKE NEWS
Score: -1.1309
--------------------------------------------------


In [13]:
test_results = []

for news in test_news:
    
    result, score = predict_news_with_score(news)
    
    test_results.append({
        "News": news.strip(),
        "Prediction": result,
        "Decision Score": round(score, 4)
    })

results_df = pd.DataFrame(test_results)

results_df

,News,Prediction,Decision Score
0,The national education department announced\n ...,FAKE NEWS,-0.6756
1,Scientists have discovered a magical device\n ...,FAKE NEWS,-0.6802
2,The central bank announced changes to its\n ...,FAKE NEWS,-0.2673
3,A viral message claims that drinking a special...,FAKE NEWS,-1.1309


In [14]:
joblib.dump(
    svm_model,
    "../model/final_fake_news_model.pkl"
)

joblib.dump(
    tfidf,
    "../model/final_tfidf_vectorizer.pkl"
)

print("Final deployment files saved!")

Final deployment files saved!


In [15]:
loaded_model = joblib.load(
    "../model/final_fake_news_model.pkl"
)

loaded_vectorizer = joblib.load(
    "../model/final_tfidf_vectorizer.pkl"
)

print("Saved model loaded successfully!")

Saved model loaded successfully!


In [16]:
news = """
The government announced a new initiative
to improve access to digital education.
"""

cleaned = clean_text(news)

vector = loaded_vectorizer.transform(
    [cleaned]
)

prediction = loaded_model.predict(
    vector
)[0]

if prediction == 0:
    print("Prediction: FAKE NEWS")
else:
    print("Prediction: REAL NEWS")

Prediction: FAKE NEWS
